In [ ]:
### Infer with your own fine-tuned Qwen 2.5 model for Polymer property prediction

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftConfig, PeftModel
from transformers import AutoTokenizer

In [2]:
BASE_MODEL_ID   = "unsloth/Qwen2.5-32B"          # your base
LORA_ADAPTER_ID = ""         # <<— change this

In [3]:
HF_TOKEN = ""  # <<— change this

In [4]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,  #your fine tuned model
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
    token          = HF_TOKEN,     # only needed if the base is gated/private
)

==((====))==  Unsloth 2025.9.4: Fast Qwen2 patching. Transformers: 4.56.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 4/4 [01:21<00:00, 20.31s/it]


In [5]:
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID)

In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
print("eos checked")

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Based on the given SMILES string, predict the monomer's relevant properties. Answer the task in this format: The monomer compound has sigma of ? GM at 780 nm, maximum sigma of ? GM, ISC of ? eV, boiling point of ? °C, logP of ?, aromaticity of ?, solubility of ? ug/mol, molecular weight of ? g/mol.", # instruction
        "CC12NC1C1C(C#N)C21", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)
print("sample checked")

In [9]:
import pandas as pd
# === Load SMILES ===
file_path = "3000.csv"
smiles_df = pd.read_csv(file_path)
all_outputs = []

print("3000.csv loaded")

3000.csv loaded


In [ ]:
for i, smi in enumerate(smiles_df["SMILES"].head(3000)):  # just first 100
    prompt = alpaca_prompt.format(
        "Based on the given SMILES string, predict the monomer's relevant properties. Answer the task in this format: The monomer compound has sigma of ? GM at 780 nm, maximum sigma of ? GM, ISC of ? eV, boiling point of ? °C, logP of ?, aromaticity of ?, solubility of ? ug/mol, molecular weight of ? g/mol.",
        smi,
        ""  # leave blank for generation
    )
    print(smi)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    all_outputs.append(decoded)

    print(f"Completed {i+1}/3000")  # progress bar in notebook

# === Save results ===
with open("3000outputs.txt", "w", encoding="utf-8") as f:
    for out in all_outputs:
        f.write(out + "\n\n")